# Agent-to-Agent (A2A) Handoff
## A Hands-On Workshop

**Agent-to-Agent (A2A) handoff** is the backbone of multi-agent systems: the mechanism by which one agent passes a structured task to another, receives the result, and incorporates it into a final answer. Getting this interface right — typed, validated, serializable — is the difference between a brittle prototype and a production-ready pipeline.

By the end of this session you will understand *why* typed handoffs matter, *how* the A2A protocol structures agent cards and task delegation, and *how* to build a planner → executor → synthesizer pipeline using LangGraph with Pydantic contracts.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — What is A2A and why typed handoffs exist |
| 2 | **Pydantic Contracts** — Defining the AgentTask schema |
| 3 | **Agent Cards** — How agents advertise their capabilities |
| 4 | **Planner Agent** — Decomposing a request into a structured task |
| 5 | **Executor Agent** — Receiving and completing the task |
| 6 | **Full Pipeline** — Planner → Executor → Synthesizer in LangGraph |
| 7 | **Task Lifecycle** — States, errors, and conditional routing |
| ★ | **Extensions** — Async concurrent execution (bonus) |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing the project's `requirements.txt`
- An `OPENAI_API_KEY` in `.env` (or Colab Secrets)

### Key References
> Google A2A Protocol Specification (2025). *Agent2Agent: An open protocol enabling AI agent interoperability.* https://github.com/google-deepmind/a2a  
> Yao, S., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629  
> Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation.* https://arxiv.org/abs/2308.08155  
> LangGraph multi-agent documentation: https://langchain-ai.github.io/langgraph/concepts/multi_agent/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "pydantic",
            "python-dotenv",
            "typing_extensions",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import asyncio
import json
import os
import time
from typing import Literal, Optional

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from pydantic import BaseModel, Field, ValidationError, field_validator
from typing_extensions import TypedDict

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
api_key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(api_key) and api_key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — What Is A2A and Why Typed Handoffs?

---

### The Problem

When agents collaborate, they need to pass work to each other. The naive approach is to pass a plain string:

```python
planner_output = "Research vector databases and write a summary"
executor_input = planner_output  # just a string — what fields exist?
```

This breaks down immediately: the executor doesn't know if it should produce bullet points or prose, how long the output should be, or what background context the planner already had. Every agent that receives a raw string must *parse* it, making the system fragile.

**A2A (Agent-to-Agent)** is Google's open protocol (2025) that solves this with:
1. **Agent Cards** — a JSON manifest that declares what an agent can do, its endpoints, and its capabilities
2. **Structured Tasks** — typed messages (not strings) that flow between agents
3. **Task lifecycle states** — submitted → working → completed / failed
4. **Push and pull notifications** — agents can subscribe to results or poll for them

---

### Three Communication Approaches Compared

| Approach | Schema? | Validated? | Debuggable? | Best for |
|----------|---------|-----------|-------------|----------|
| **Raw string** | None | No | Hard | Quick prototypes only |
| **Dict** | Implicit | No | OK | Internal use, no external agents |
| **Pydantic model** | Explicit | Yes | Yes (serializable) | Production A2A pipelines |
| **A2A protocol (full)** | JSON Schema | Yes | Yes (Agent Card) | Cross-org agent networks |

---

### A2A vs MCP vs Function Calling

| | **A2A** | **MCP** | **Function Calling** |
|-|---------|---------|---------------------|
| **Purpose** | Agent ↔ Agent tasks | Model ↔ Tool/Resource | Model → Tool invocation |
| **Who calls?** | One agent calls another | Model calls a tool server | Model calls a local function |
| **Schema** | Agent Card (JSON) | Tool manifest (JSON Schema) | Function signature |
| **Async support** | Yes (push/pull) | No (synchronous) | No |
| **Discovery** | Agent Card URL | MCP server config | In-prompt definition |
| **Result routing** | Task → next agent | Returns to model | Returns to model |

> **Rule of thumb:** Use function calling when a model needs one tool. Use MCP when a model needs many tools from a server. Use A2A when *agents need to delegate entire tasks to other agents*.

---

### A2A Handoff Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                          A2A HANDOFF PIPELINE                               │
└─────────────────────────────────────────────────────────────────────────────┘

  User Request
       │
       ▼
  ┌────────────────────┐   Agent Card lookup   ┌─────────────────────────┐
  │   PLANNER AGENT    │─────────────────────▶ │    Agent Card (JSON)    │
  │  (Client Agent)    │                        │  name, description,     │
  │                    │◀─────────────────────  │  capabilities, skills   │
  └────────────────────┘   card returned        └─────────────────────────┘
       │
       │  Structured Task (AgentTask)
       │  ┌──────────────────────────────┐
       │  │ task_id:        "vec-db-001" │
       │  │ task_type:      "research"   │
       │  │ instruction:    "Explain..." │
       │  │ context:        "User needs" │
       │  │ expected_output: "3-para..." │
       │  └──────────────────────────────┘
       │
       ▼
  ┌────────────────────┐
  │   EXECUTOR AGENT   │  status: submitted → working → completed
  │  (Remote Agent)    │
  └────────────────────┘
       │
       │  Execution Result (text)
       │
       ▼
  ┌────────────────────┐
  │  SYNTHESIZER AGENT │  Merges original request + execution result
  │  (Client Agent)    │
  └────────────────────┘
       │
       ▼
  Polished Final Answer
```

> **Source:** Architecture based on Google's A2A protocol specification (2025). https://github.com/google-deepmind/a2a

---

## Part 2 — Pydantic Contracts: The AgentTask Schema

---

### Why Pydantic?

A2A tasks are typed messages. In Python, **Pydantic** is the standard library for declaring those types:

- `Field(description=...)` annotates each field — LLMs use these descriptions when generating structured output via `with_structured_output`
- `.model_dump()` serializes to a dict for storage in LangGraph state
- `@field_validator` lets you add business-rule validation (e.g., `task_type` must be one of three values)
- The schema is self-documenting: any agent receiving an `AgentTask` knows exactly what fields exist

---

### AgentTask Field Reference

| Field | Type | Purpose | Example |
|-------|------|---------|--------|
| `task_id` | `str` | Unique slug for tracing and deduplication | `"vec-db-research-001"` |
| `task_type` | `str` | Classification for routing and logging | `"research"` |
| `instruction` | `str` | The actual directive the executor must follow | `"Explain the key benefits..."` |
| `context` | `str` | Background the planner knows that the executor needs | `"User is a backend engineer"` |
| `expected_output` | `str` | Contract for what the executor must produce | `"3-paragraph summary with examples"` |

In [ ]:
# ─── 2-A: Define the AgentTask contract ──────────────────────────────────────
# This is the typed handoff schema that flows from Planner to Executor.
# Field descriptions guide the LLM when generating structured output.


class AgentTask(BaseModel):
    """Structured handoff from Planner Agent to Executor Agent.

    This Pydantic model is the A2A task contract — the formal interface
    between agents. It replaces raw strings with a validated, serializable
    schema that both sides of the handoff agree on.
    """

    task_id: str = Field(
        description="Unique identifier for this task — a short kebab-case slug"
    )
    task_type: str = Field(
        description="Task classification: must be 'research', 'analysis', or 'synthesis'"
    )
    instruction: str = Field(
        description="Clear, unambiguous directive for the executor agent"
    )
    context: str = Field(
        description="Background context the executor needs to complete the task well"
    )
    expected_output: str = Field(
        description="Precise description of what the executor should produce and how long"
    )


# Inspect the generated JSON schema — this is what the LLM sees
print("AgentTask JSON Schema (what the LLM uses to generate structured output):")
print(json.dumps(AgentTask.model_json_schema(), indent=2))

In [ ]:
# ─── 2-B: Create and inspect a task manually ─────────────────────────────────
# Before wiring up LLMs, understand what a valid task looks like.

example_task = AgentTask(
    task_id="vec-db-research-001",
    task_type="research",
    instruction="Explain the key benefits of vector databases compared to traditional relational databases.",
    context="The user is a backend engineer evaluating data storage options for a semantic search feature.",
    expected_output="A 3-paragraph summary with one concrete example per benefit, written for a technical audience.",
)

print("=== AgentTask (Pydantic model) ===")
print(f"task_id        : {example_task.task_id}")
print(f"task_type      : {example_task.task_type}")
print(f"instruction    : {example_task.instruction[:80]}...")
print(f"context        : {example_task.context[:80]}...")
print(f"expected_output: {example_task.expected_output[:80]}...")
print()
print("=== Serialized to dict (stored in LangGraph state) ===")
task_dict = example_task.model_dump()
print(json.dumps(task_dict, indent=2))

In [ ]:
# ─── 2-C: Validation in action ────────────────────────────────────────────────
# Pydantic validates on creation. This is what catches malformed LLM output
# before it propagates to the next agent.

# Valid task — all required fields present
try:
    valid = AgentTask(
        task_id="test-001",
        task_type="analysis",
        instruction="Compare the CAP theorem trade-offs.",
        context="Database architecture workshop.",
        expected_output="Bullet-point comparison, 200 words.",
    )
    print(f"Valid task created: {valid.task_id}")
except ValidationError as e:
    print(f"Unexpected validation failure: {e}")

print()

# Missing required fields — Pydantic raises ValidationError immediately
try:
    invalid = AgentTask(
        task_id="test-002",
        task_type="analysis",
        # missing: instruction, context, expected_output
    )
    print("ERROR: should have raised ValidationError")
except ValidationError as e:
    print("ValidationError caught (expected):")
    for error in e.errors():
        print(f"  [{error['loc'][0]}] {error['msg']}")

print()
print("Observation: Pydantic catches missing fields at creation time,")
print("not when the executor tries to access task['instruction'] later.")

---

## Part 3 — Agent Cards: How Agents Advertise Capabilities

---

### What Is an Agent Card?

In the full A2A protocol, every agent publishes a **well-known JSON document** at `/.well-known/agent.json` — its **Agent Card**. A client agent fetches this URL before delegating a task, so it knows:

- What the remote agent can do (skills)
- How to authenticate
- What input format it accepts
- What output format it returns

This is A2A's equivalent of an API specification — it enables agents to *discover* each other at runtime, not just at development time.

---

### Agent Card Fields

| Field | Required | Purpose | Example |
|-------|----------|---------|--------|
| `name` | Yes | Human-readable agent name | `"ResearchExecutor"` |
| `description` | Yes | What this agent does | `"Performs in-depth research on technical topics"` |
| `url` | Yes | Base endpoint for task submission | `"https://agents.example.com/research"` |
| `version` | Yes | Semantic version | `"1.0.0"` |
| `capabilities` | Yes | Feature flags (streaming, push, state) | `{"streaming": true}` |
| `skills` | Yes | List of task types this agent handles | `[{"id": "research", "name": "Deep Research"}]` |
| `authentication` | No | Auth scheme | `{"schemes": ["bearer"]}` |
| `defaultInputModes` | No | Accepted input content types | `["text/plain"]` |
| `defaultOutputModes` | No | Output content types | `["text/markdown"]` |

---

### Task Lifecycle States

| State | Meaning | Next States |
|-------|---------|-------------|
| `submitted` | Task received by remote agent | `working` |
| `working` | Remote agent is executing | `completed`, `failed`, `input-required` |
| `input-required` | Agent needs clarification from client | `working` |
| `completed` | Task finished successfully | (terminal) |
| `failed` | Task could not be completed | (terminal) |
| `canceled` | Client canceled the task | (terminal) |

---

### Push vs Pull Notification Patterns

| Pattern | How it works | Best for |
|---------|-------------|----------|
| **Synchronous (pull)** | Client blocks until executor returns | Short tasks, simple pipelines |
| **Polling (pull)** | Client polls status endpoint until `completed` | Long tasks, unreliable connections |
| **Webhook (push)** | Executor POSTs result to client's callback URL | Async pipelines, parallel tasks |
| **SSE stream (push)** | Executor streams partial results as Server-Sent Events | Long tasks needing live updates |

In [ ]:
# ─── 3-A: Simulate an Agent Card ─────────────────────────────────────────────
# In production this JSON lives at /.well-known/agent.json on the remote host.
# Here we define it inline to see the full structure.

EXECUTOR_AGENT_CARD = {
    "name": "ResearchExecutorAgent",
    "description": "Executes research, analysis, and synthesis tasks. Returns structured Markdown results.",
    "url": "https://agents.example.com/executor",
    "version": "1.0.0",
    "capabilities": {
        "streaming": False,
        "pushNotifications": False,
        "stateTransitionHistory": True,
    },
    "defaultInputModes": ["application/json"],
    "defaultOutputModes": ["text/markdown"],
    "authentication": {"schemes": ["bearer"]},
    "skills": [
        {
            "id": "research",
            "name": "Deep Research",
            "description": "In-depth research on technical, scientific, or business topics",
            "tags": ["research", "information-retrieval", "summarization"],
            "examples": ["Explain the benefits of vector databases"],
        },
        {
            "id": "analysis",
            "name": "Technical Analysis",
            "description": "Analyzes trade-offs, compares approaches, or evaluates options",
            "tags": ["analysis", "comparison", "evaluation"],
            "examples": ["Compare CAP theorem trade-offs"],
        },
        {
            "id": "synthesis",
            "name": "Content Synthesis",
            "description": "Synthesizes multiple sources into a coherent narrative",
            "tags": ["synthesis", "writing", "summarization"],
            "examples": ["Write an executive summary from research notes"],
        },
    ],
}

print("Agent Card for ResearchExecutorAgent:")
print(json.dumps(EXECUTOR_AGENT_CARD, indent=2))

In [ ]:
# ─── 3-B: A planner uses the Agent Card to choose the right executor ──────────
# In a real A2A system the planner fetches the card via HTTP.
# This simulates the skill-matching logic that would happen at runtime.


def find_matching_skill(agent_card: dict, task_type: str) -> dict | None:
    """Match a task_type to an agent's declared skills."""
    for skill in agent_card.get("skills", []):
        if skill["id"] == task_type:
            return skill
    return None


def simulate_task_submission(agent_card: dict, task: AgentTask) -> dict:
    """Simulate submitting a task to a remote agent via A2A protocol."""
    skill = find_matching_skill(agent_card, task.task_type)
    if skill is None:
        return {
            "id": task.task_id,
            "status": {"state": "failed"},
            "error": f"Agent does not support task_type='{task.task_type}'",
        }
    return {
        "id": task.task_id,
        "status": {"state": "submitted"},
        "skill_matched": skill["name"],
        "endpoint": agent_card["url"],
    }


# Test with a supported task type
submission = simulate_task_submission(EXECUTOR_AGENT_CARD, example_task)
print("Supported task:")
print(json.dumps(submission, indent=2))

print()

# Test with an unsupported task type
unsupported_task = AgentTask(
    task_id="test-003",
    task_type="code-generation",  # not in this agent's skills
    instruction="Write a Python function",
    context="Workshop demo",
    expected_output="A Python function",
)
failed = simulate_task_submission(EXECUTOR_AGENT_CARD, unsupported_task)
print("Unsupported task type:")
print(json.dumps(failed, indent=2))

---

## Part 4 — Planner Agent: Decomposing Requests into Structured Tasks

---

### How `with_structured_output` Works

LangChain's `ChatOpenAI.with_structured_output(AgentTask)` does three things:

1. **Generates a tool schema** from the Pydantic model's field names and `Field(description=...)` annotations
2. **Forces the LLM** to respond with a valid JSON object matching that schema via tool calling
3. **Parses and validates** the JSON response back into an `AgentTask` instance — raising `ValidationError` if invalid

```
Planner LLM:
   input:  "Research vector databases"
   output: AgentTask(task_id="vec-db-001", task_type="research", ...)
            (not a string — a validated Pydantic object)
```

### LangGraph State

The graph state (`A2AState`) threads through all nodes. Each node reads from it and returns a partial dict of updates:

| Field | Type | Set by | Read by |
|-------|------|--------|--------|
| `original_request` | `str` | User | Planner, Synthesizer |
| `task_dict` | `dict \| None` | Planner | Executor, Synthesizer |
| `execution_result` | `str` | Executor | Synthesizer |
| `final_synthesis` | `str` | Synthesizer | Output |

In [ ]:
# ─── 4-A: Set up the LLMs ────────────────────────────────────────────────────
# Two LLM instances:
#   llm           — general-purpose, used by executor and synthesizer
#   planner_llm   — same model, wired to return AgentTask via tool calling

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# with_structured_output generates the JSON schema from AgentTask's fields
# and instructs the model to call it as a tool, returning a validated instance.
planner_llm = llm.with_structured_output(AgentTask)

print("LLM ready: gpt-4o-mini")
print("planner_llm wired to return AgentTask via structured output")
print()
print("Fields the LLM must populate:")
for field_name, field_info in AgentTask.model_fields.items():
    print(f"  {field_name:20} — {field_info.description}")

In [ ]:
# ─── 4-B: Define state and prompts ───────────────────────────────────────────


class A2AState(TypedDict):
    original_request: str
    task_dict: Optional[dict]  # AgentTask serialized via .model_dump()
    execution_result: str
    final_synthesis: str


PLANNER_PROMPT = """You are a planning agent. Given a user's request, create a structured task
for a specialized executor agent.

Request: {request}

Create a task with:
- task_id: short kebab-case slug (e.g. 'vec-db-research-001')
- task_type: one of 'research', 'analysis', or 'synthesis'
- instruction: a clear, unambiguous directive for the executor
- context: background the executor needs to do a good job
- expected_output: precise description of what to produce and in what format"""

EXECUTOR_PROMPT = """You are an expert executor agent. Complete the task below thoroughly.

Task ID: {task_id}
Task: {instruction}
Context: {context}
Expected Output: {expected_output}

Provide a clear, well-structured response that exactly matches the expected output format."""

SYNTHESIZER_PROMPT = """You are a planning agent finalizing a response for the user.

Original Request: {original_request}
Task Delegated: {task_instruction}
Executor Result:\n{execution_result}

Synthesize a polished, complete final answer that directly addresses the original request.
Integrate the executor's result naturally — do not reference internal task IDs or agent names."""

print("A2AState fields:", list(A2AState.__annotations__.keys()))
print("Three prompts defined: PLANNER_PROMPT, EXECUTOR_PROMPT, SYNTHESIZER_PROMPT")

In [ ]:
# ─── 4-C: Run the Planner agent in isolation ──────────────────────────────────
# Always test individual nodes before composing the full graph.
# This shows exactly what the Planner produces for a given request.

test_request = "Research the key benefits of vector databases and write a summary."

planner_prompt_text = PLANNER_PROMPT.format(request=test_request)
planner_output: AgentTask = planner_llm.invoke([HumanMessage(content=planner_prompt_text)])

print("Planner output (AgentTask):")
print(f"  task_id        : {planner_output.task_id}")
print(f"  task_type      : {planner_output.task_type}")
print(f"  instruction    : {planner_output.instruction}")
print(f"  context        : {planner_output.context}")
print(f"  expected_output: {planner_output.expected_output}")
print()
print("Type:", type(planner_output).__name__)
print("Is valid AgentTask:", isinstance(planner_output, AgentTask))
print()
print("Serialized dict (ready for LangGraph state):")
print(json.dumps(planner_output.model_dump(), indent=2))

---

## Part 5 — Executor Agent: Receiving and Completing the Task

---

### The Handoff Contract in Action

The executor agent receives a `task_dict` from the graph state — the Planner's `AgentTask` serialized via `.model_dump()`. It reads the dict fields, formats the prompt, and returns the result:

```python
state["task_dict"] = {
    "task_id": "vec-db-research-001",
    "task_type": "research",
    "instruction": "...",
    "context": "...",
    "expected_output": "..."
}
```

**Why dict in state (not the Pydantic model itself)?**  
LangGraph state must be JSON-serializable for persistence, checkpointing, and replay. Pydantic models are not JSON-serializable by default — `.model_dump()` converts them to a plain dict which is. Reconstruct the model from the dict at any time via `AgentTask(**state['task_dict'])`.

In [ ]:
# ─── 5-A: Run the Executor agent in isolation ─────────────────────────────────
# Feed it the planner's task_dict and inspect the result.

task_dict = planner_output.model_dump()  # from Part 4-C

executor_prompt_text = EXECUTOR_PROMPT.format(
    task_id=task_dict["task_id"],
    instruction=task_dict["instruction"],
    context=task_dict["context"],
    expected_output=task_dict["expected_output"],
)

executor_response = llm.invoke([HumanMessage(content=executor_prompt_text)])

print(f"Executor result ({len(executor_response.content)} chars):")
print("─" * 60)
print(executor_response.content[:600])
if len(executor_response.content) > 600:
    print(f"... [{len(executor_response.content) - 600} more chars]")

In [ ]:
# ─── 5-B: Synthesizer agent in isolation ─────────────────────────────────────
# The synthesizer takes the original user request + executor result
# and produces the final polished answer the user actually receives.

synthesizer_prompt_text = SYNTHESIZER_PROMPT.format(
    original_request=test_request,
    task_instruction=task_dict["instruction"],
    execution_result=executor_response.content,
)

synthesizer_response = llm.invoke([HumanMessage(content=synthesizer_prompt_text)])

print("Synthesized final answer:")
print("─" * 60)
print(synthesizer_response.content[:600])
if len(synthesizer_response.content) > 600:
    print(f"... [{len(synthesizer_response.content) - 600} more chars]")

---

## Part 6 — Full Pipeline: Planner → Executor → Synthesizer in LangGraph

---

### Graph Design

```
START
  │
  ▼
[planner]      ← creates AgentTask via with_structured_output, serializes to state
  │
  ▼
[executor]     ← reads task_dict from state, runs LLM, writes execution_result
  │
  ▼
[synthesizer]  ← reads original_request + execution_result, writes final_synthesis
  │
  ▼
 END
```

Each node is a **pure function**: `(state) -> dict`. It reads from the shared state and returns only the keys it updates. LangGraph merges the returned dict into the state before calling the next node.

This design means:
- Nodes are independently testable (as shown in Parts 4 and 5 above)
- The graph can be extended with conditional edges or parallel branches without rewriting existing nodes
- The full state at any point is inspectable — useful for debugging

In [ ]:
# ─── 6-A: Define the three agent nodes ───────────────────────────────────────


def planner_agent(state: A2AState) -> dict:
    """Agent A (Planner): decomposes the user request into a typed AgentTask.

    Uses with_structured_output so the LLM is forced to return a validated
    AgentTask object. Serializes to dict via .model_dump() for LangGraph state.
    """
    prompt = PLANNER_PROMPT.format(request=state["original_request"])
    task: AgentTask = planner_llm.invoke([HumanMessage(content=prompt)])
    print(f"  [planner] created task: {task.task_id} ({task.task_type})")
    print(f"  [planner] instruction: {task.instruction[:80]}...")
    return {"task_dict": task.model_dump()}


def executor_agent(state: A2AState) -> dict:
    """Agent B (Executor): receives the AgentTask dict and completes it.

    This is the 'remote agent' in A2A terms — it could live at any URL
    and be called over HTTP in a production system.
    """
    task = state["task_dict"]
    prompt = EXECUTOR_PROMPT.format(
        task_id=task["task_id"],
        instruction=task["instruction"],
        context=task["context"],
        expected_output=task["expected_output"],
    )
    result = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [executor] completed: {len(result.content)} chars")
    return {"execution_result": result.content}


def synthesizer_agent(state: A2AState) -> dict:
    """Agent A (Synthesizer): merges executor result with original request.

    Note: the same Planner agent takes a second role — a common pattern
    where the client agent both plans AND synthesizes, while a specialized
    remote agent handles execution.
    """
    task = state["task_dict"]
    prompt = SYNTHESIZER_PROMPT.format(
        original_request=state["original_request"],
        task_instruction=task["instruction"],
        execution_result=state["execution_result"],
    )
    result = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [synthesizer] final answer: {len(result.content)} chars")
    return {"final_synthesis": result.content}


print("Three agent nodes defined: planner_agent, executor_agent, synthesizer_agent")

In [ ]:
# ─── 6-B: Compile the LangGraph workflow ─────────────────────────────────────


def create_workflow():
    graph = StateGraph(A2AState)

    # Register nodes
    graph.add_node("planner", planner_agent)
    graph.add_node("executor", executor_agent)
    graph.add_node("synthesizer", synthesizer_agent)

    # Wire edges: linear flow planner → executor → synthesizer → END
    graph.set_entry_point("planner")
    graph.add_edge("planner", "executor")
    graph.add_edge("executor", "synthesizer")
    graph.add_edge("synthesizer", END)

    return graph.compile()


app = create_workflow()
print("Workflow compiled successfully.")
print()
print("Graph nodes:", list(app.get_graph().nodes.keys()))
print("Graph edges:")
for edge in app.get_graph().edges:
    print(f"  {edge[0]} → {edge[1]}")

In [ ]:
# ─── 6-C: Run the full pipeline ───────────────────────────────────────────────

SAMPLE_REQUESTS = [
    "Research the key benefits of vector databases and write a summary.",
    "Explain the CAP theorem and its practical implications for distributed systems.",
]

for request in SAMPLE_REQUESTS:
    print(f"\nREQUEST: {request}")
    print("═" * 70)

    result = app.invoke(
        {
            "original_request": request,
            "task_dict": None,
            "execution_result": "",
            "final_synthesis": "",
        }
    )

    task = result["task_dict"]
    print()
    print(f"Task ID    : {task['task_id']}")
    print(f"Task Type  : {task['task_type']}")
    print(f"Instruction: {task['instruction'][:100]}...")
    print(f"Exec chars : {len(result['execution_result'])}")
    print()
    print("Final Answer (first 400 chars):")
    print("─" * 60)
    print(result["final_synthesis"][:400])
    print()

---

## Part 7 — Task Lifecycle: Conditional Routing and Retry

---

### Adding a QA Gate

Production A2A pipelines often add a **quality gate** between executor and synthesizer. If the executor's output is too short, off-topic, or fails a heuristic check, the task routes back for revision rather than synthesizing a bad result.

LangGraph supports this via **conditional edges** — a router function that inspects state and returns the name of the next node:

```python
def route_after_executor(state: A2AState) -> str:
    if len(state["execution_result"]) < 200:
        return "executor"      # retry
    return "synthesizer"       # proceed

graph.add_conditional_edges("executor", route_after_executor)
```

The graph with a conditional retry edge looks like:

```
START → [planner] → [executor] ──(result ok)──▶ [synthesizer] → END
                        ▲                                  
                        └──(result too short, retry < MAX)──┘
```

In [ ]:
# ─── 7-A: Extended AgentTask with priority and strict type validation ─────────
# Demonstrates Literal types + @field_validator for business-rule enforcement.


class AgentTaskExtended(BaseModel):
    """Extended A2A task with priority, retry tracking, and strict validation."""

    task_id: str = Field(description="Unique kebab-case task identifier")
    task_type: Literal["research", "analysis", "synthesis"] = Field(
        description="Task classification — must be research, analysis, or synthesis"
    )
    instruction: str = Field(description="Clear directive for the executor")
    context: str = Field(description="Background context for the executor")
    expected_output: str = Field(description="Description of the expected result")
    priority: Literal["low", "medium", "high"] = Field(
        default="medium",
        description="Task priority: low, medium, or high",
    )
    max_retries: int = Field(
        default=1,
        description="Maximum executor retries on failure",
    )

    @field_validator("task_id")
    @classmethod
    def task_id_must_be_slug(cls, v: str) -> str:
        """Enforce kebab-case slug format for task IDs."""
        import re

        if not re.match(r"^[a-z0-9][a-z0-9-]*[a-z0-9]$", v):
            raise ValueError(
                f"task_id must be kebab-case (e.g. 'my-task-001'), got: '{v}'"
            )
        return v


# Valid — all fields present, slug is valid
valid_ext = AgentTaskExtended(
    task_id="cap-analysis-001",
    task_type="analysis",
    instruction="Analyze the CAP theorem trade-offs.",
    context="Distributed systems course.",
    expected_output="300-word analysis with CP vs AP examples.",
    priority="high",
)
print(f"Valid extended task: {valid_ext.task_id} (priority={valid_ext.priority}, max_retries={valid_ext.max_retries})")

print()

# Invalid task_id — @field_validator fires
try:
    AgentTaskExtended(
        task_id="Bad Task ID!!",
        task_type="research",
        instruction="Test",
        context="Test",
        expected_output="Test",
    )
except ValidationError as e:
    print("ValidationError on bad task_id (expected):")
    for err in e.errors():
        print(f"  [{err['loc'][0]}] {err['msg']}")

In [ ]:
# ─── 7-B: Conditional routing — retry if executor output is too short ─────────


class A2AStateWithRetry(TypedDict):
    original_request: str
    task_dict: Optional[dict]
    execution_result: str
    final_synthesis: str
    retry_count: int


def executor_agent_v2(state: A2AStateWithRetry) -> dict:
    """Executor with retry count tracking."""
    task = state["task_dict"]
    prompt = EXECUTOR_PROMPT.format(
        task_id=task["task_id"],
        instruction=task["instruction"],
        context=task["context"],
        expected_output=task["expected_output"],
    )
    result = llm.invoke([HumanMessage(content=prompt)])
    retry = state.get("retry_count", 0)
    print(f"  [executor] attempt {retry + 1}: {len(result.content)} chars")
    return {"execution_result": result.content, "retry_count": retry + 1}


def route_after_executor(state: A2AStateWithRetry) -> str:
    """Route to synthesizer if result is good; retry executor if too short."""
    result_len = len(state.get("execution_result", ""))
    retry_count = state.get("retry_count", 0)
    MAX_RETRIES = 2

    if result_len < 200 and retry_count < MAX_RETRIES:
        print(f"  [router] result too short ({result_len} chars), retrying...")
        return "executor"

    print(f"  [router] result accepted ({result_len} chars), proceeding to synthesizer")
    return "synthesizer"


def create_workflow_with_retry():
    graph = StateGraph(A2AStateWithRetry)
    graph.add_node("planner", planner_agent)
    graph.add_node("executor", executor_agent_v2)
    graph.add_node("synthesizer", synthesizer_agent)
    graph.set_entry_point("planner")
    graph.add_edge("planner", "executor")
    graph.add_conditional_edges(
        "executor",
        route_after_executor,
        {"executor": "executor", "synthesizer": "synthesizer"},
    )
    graph.add_edge("synthesizer", END)
    return graph.compile()


app_with_retry = create_workflow_with_retry()
print("Retry workflow compiled.")
print("Graph edges (including conditional):")
for edge in app_with_retry.get_graph().edges:
    print(f"  {edge[0]} → {edge[1]}")

In [ ]:
# ─── 7-C: Run the retry workflow ──────────────────────────────────────────────

print("Running pipeline with QA gate (retry if result < 200 chars):\n")

result_retry = app_with_retry.invoke(
    {
        "original_request": "Explain the CAP theorem and its practical implications.",
        "task_dict": None,
        "execution_result": "",
        "final_synthesis": "",
        "retry_count": 0,
    }
)

print()
print(f"Retries used  : {result_retry['retry_count'] - 1}")
print(f"Exec chars    : {len(result_retry['execution_result'])}")
print(f"Final chars   : {len(result_retry['final_synthesis'])}")
print()
print("Final Answer (first 400 chars):")
print("─" * 60)
print(result_retry["final_synthesis"][:400])

---

## Exercises

Work through these in order — each builds on the previous.

### Exercise 1 — Add a `priority` field to AgentTask

Extend the base `AgentTask` class with a `priority: Literal["low", "medium", "high"]` field (default `"medium"`). Update `PLANNER_PROMPT` to instruct the LLM to assign `"high"` priority for requests that mention urgency or production issues. Run the workflow and verify that urgent requests get `"high"` priority.

### Exercise 2 — Add a field validator for `task_type`

Add a `@field_validator("task_type")` to the base `AgentTask` that raises `ValueError` if the value is not in `["research", "analysis", "synthesis"]`. Then deliberately invoke the planner with a prompt that might produce an invalid `task_type` (e.g., tell it to classify tasks as `"code-generation"`). Observe what happens when `with_structured_output` validation fails.

### Exercise 3 — Multi-task decomposition

The current planner creates exactly one `AgentTask`. Extend the state and planner to decompose a complex request into **two tasks** run sequentially: a `research` task followed by an `analysis` task. The synthesizer should merge both results. Modify `A2AState` to hold a list of task dicts and add a second executor node.

### Exercise 4 — Async concurrent execution

Convert the executor node to use `await llm.ainvoke()` and run two independent requests concurrently with `asyncio.gather()`. Compare the wall-clock time of sequential vs concurrent execution.

In [ ]:
# ── Exercise Starters ─────────────────────────────────────────────────────────

# ── Exercise 1: priority field ────────────────────────────────────────────────
class AgentTaskEx1(BaseModel):
    task_id: str = Field(description="Unique identifier for this task")
    task_type: str = Field(description="Type: research, analysis, or synthesis")
    instruction: str = Field(description="Clear instruction for the executor")
    context: str = Field(description="Background context to help executor")
    expected_output: str = Field(description="What the executor should produce")
    # TODO: add priority field
    # priority: Literal["low", "medium", "high"] = Field(
    #     default="medium",
    #     description="...",
    # )


PLANNER_PROMPT_EX1 = """
You are a planning agent. Create a structured task for this request.
# TODO: add instruction about when to assign high priority

Request: {request}
"""


# ── Exercise 2: field validator for task_type ─────────────────────────────────
class AgentTaskEx2(BaseModel):
    task_id: str = Field(description="Unique identifier for this task")
    task_type: str = Field(description="Type: research, analysis, or synthesis")
    instruction: str = Field(description="Clear instruction for the executor")
    context: str = Field(description="Background context to help executor")
    expected_output: str = Field(description="What the executor should produce")

    # TODO: add @field_validator("task_type") that enforces allowed values
    # @field_validator("task_type")
    # @classmethod
    # def task_type_must_be_valid(cls, v: str) -> str:
    #     allowed = {"research", "analysis", "synthesis"}
    #     ...


# ── Exercise 3: multi-task decomposition ──────────────────────────────────────
class A2AStateMultiTask(TypedDict):
    original_request: str
    task_dicts: list  # TODO: list of AgentTask dicts
    execution_results: list
    final_synthesis: str


# ── Exercise 4: async execution ───────────────────────────────────────────────
# async def executor_agent_async(state: A2AState) -> dict:
#     task = state["task_dict"]
#     prompt = EXECUTOR_PROMPT.format(
#         task_id=task["task_id"],
#         instruction=task["instruction"],
#         context=task["context"],
#         expected_output=task["expected_output"],
#     )
#     # TODO: use await llm.ainvoke([HumanMessage(content=prompt)])
#     pass

print("Exercise starters loaded — fill in the TODO sections above.")

---

## Part 8 ★ — Async Concurrent Execution (Bonus)

---

### Why Async?

In the synchronous pipeline, requests run one at a time — the notebook waits for each LLM call to complete before starting the next. For multiple independent tasks, this is wasteful.

All LangChain LLMs support `await llm.ainvoke(...)`. In a Jupyter notebook the event loop is already running, so `await` works at the top level:

```python
results = await asyncio.gather(
    llm.ainvoke([HumanMessage(content=prompt_a)]),
    llm.ainvoke([HumanMessage(content=prompt_b)]),
)
# both calls run concurrently — total time ≈ max(latency_a, latency_b)
```

In a regular Python script, wrap with `asyncio.run(...)` instead.

In [ ]:
# ─── 8-A: Async concurrent execution — sequential vs concurrent timing ────────

TASK_A_PROMPT = EXECUTOR_PROMPT.format(
    task_id="async-task-a",
    instruction="Explain the key benefits of vector databases.",
    context="Backend engineering context.",
    expected_output="3-paragraph technical summary.",
)

TASK_B_PROMPT = EXECUTOR_PROMPT.format(
    task_id="async-task-b",
    instruction="Explain the CAP theorem and its trade-offs.",
    context="Distributed systems context.",
    expected_output="3-paragraph technical summary with CP vs AP examples.",
)


async def run_concurrent():
    return await asyncio.gather(
        llm.ainvoke([HumanMessage(content=TASK_A_PROMPT)]),
        llm.ainvoke([HumanMessage(content=TASK_B_PROMPT)]),
    )


# Sequential baseline
t0 = time.time()
r_a = llm.invoke([HumanMessage(content=TASK_A_PROMPT)])
r_b = llm.invoke([HumanMessage(content=TASK_B_PROMPT)])
seq_time = time.time() - t0

# Concurrent
t1 = time.time()
concurrent_results = await run_concurrent()
conc_time = time.time() - t1

total_seq_chars = len(r_a.content) + len(r_b.content)
total_conc_chars = sum(len(r.content) for r in concurrent_results)

print(f"Sequential : {seq_time:.2f}s  ({total_seq_chars} total chars)")
print(f"Concurrent : {conc_time:.2f}s  ({total_conc_chars} total chars)")
print(f"Speedup    : {seq_time / conc_time:.1f}x")
print()
print("Observation: concurrent time should approach max(latency_a, latency_b),")
print("not the sum. Real speedup varies with API concurrency limits.")

---

## What's Next?

You now understand the core A2A handoff pattern. Here is where to go deeper:

### Extend this pattern
- **Example 43 — Supervisor/Worker** (`../43-supervisor-worker/`): a supervisor agent dynamically routes tasks to multiple specialized workers based on task type — a direct extension of the conditional routing shown in Part 7.
- **Example 38 — LangGraph Command Pattern** (`../38-langgraph-command-pattern/`): `Command`-based handoffs give fine-grained control over state updates during routing, not just which node to visit next.
- **Example 6 — Multi-Agent Graph** (`../6-multi-agent-graph/`): subgraph composition — how to package an entire A2A pipeline as a reusable subgraph that another graph can call.

### Go to production
- **Full A2A protocol**: implement the actual HTTP transport with Agent Card endpoints, Bearer token auth, and JSON task messages — see https://github.com/google-deepmind/a2a for the reference implementation and SDK.
- **LangGraph Platform**: deploy this graph as a persistent, resumable service with built-in state checkpointing and human-in-the-loop approval nodes — https://langchain-ai.github.io/langgraph/cloud/
- **Observability**: enable LangSmith tracing to see every agent invocation, state transition, and token count in a dashboard — https://smith.langchain.com

### Further reading
> Google A2A Protocol (2025). *Agent2Agent specification and reference implementation.* https://github.com/google-deepmind/a2a  
> Yao, S., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629  
> Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation.* https://arxiv.org/abs/2308.08155  
> LangGraph multi-agent patterns: https://langchain-ai.github.io/langgraph/concepts/multi_agent/

---

*Workshop complete. The next step is the Supervisor/Worker example — add dynamic routing between specialized agents.*

---

## Exercise Answer Key

Use this section after attempting the exercises yourself. These are reference solutions — not the only correct answers.

### Exercise 1 — `priority` field in AgentTask

**What good output looks like:**  
The planner assigns `priority="high"` for requests containing words like "urgent", "production", "outage", or "deadline". For general research questions it defaults to `"medium"`. The key test: send the same factual request with and without urgency language and compare the priority field in each result.

**Common pitfall:** If you use `Literal["low", "medium", "high"]` as the type, Pydantic will reject any other string at validation time — but the LLM may still hallucinate values like `"critical"`. The `Literal` type catches this automatically via `with_structured_output` because the JSON schema generated from `Literal` includes an `enum` constraint that the model sees.

In [ ]:
# Answer: Exercise 1 — priority field


class AgentTaskAns1(BaseModel):
    task_id: str = Field(description="Unique kebab-case slug")
    task_type: str = Field(description="research, analysis, or synthesis")
    instruction: str = Field(description="Clear directive for the executor")
    context: str = Field(description="Background context")
    expected_output: str = Field(description="What the executor should produce")
    priority: Literal["low", "medium", "high"] = Field(
        default="medium",
        description=(
            "Task priority. Assign 'high' if the request mentions urgency, a deadline, "
            "production impact, or an outage. Use 'low' for exploratory questions."
        ),
    )


planner_with_priority = llm.with_structured_output(AgentTaskAns1)

PLANNER_PRIORITY_PROMPT = """You are a planning agent. Create a structured task for this request.
Set priority='high' for urgent or production-critical requests, 'low' for exploratory ones.

Request: {request}"""

for req in [
    "Research vector databases for our team.",
    "URGENT: our production search is down — investigate vector index corruption now.",
]:
    t = planner_with_priority.invoke(
        [HumanMessage(content=PLANNER_PRIORITY_PROMPT.format(request=req))]
    )
    print(f"Request : {req[:70]}")
    print(f"  task_id={t.task_id}  priority={t.priority}")
    print()

### Exercise 2 — Field validator for `task_type`

**What to expect:**  
When `with_structured_output` asks the model to populate `task_type` and the model returns an unlisted value (e.g. `"code-generation"`), Pydantic raises `ValidationError` during deserialization. LangChain surfaces this as a chain error — it does not silently accept invalid output.

**Better approach for strict enforcement:** use `Literal["research", "analysis", "synthesis"]` as the type directly (as shown in `AgentTaskExtended` in Part 7). Pydantic generates a JSON Schema `enum` from `Literal` which the LLM sees in its tool schema, reducing the probability of invalid values before they even reach the validator.

In [ ]:
# Answer: Exercise 2 — field validator for task_type


class AgentTaskAns2(BaseModel):
    task_id: str = Field(description="Unique identifier")
    task_type: str = Field(description="Type: research, analysis, or synthesis")
    instruction: str = Field(description="Clear instruction")
    context: str = Field(description="Background context")
    expected_output: str = Field(description="Expected output description")

    @field_validator("task_type")
    @classmethod
    def task_type_must_be_valid(cls, v: str) -> str:
        allowed = {"research", "analysis", "synthesis"}
        if v not in allowed:
            raise ValueError(
                f"task_type must be one of {sorted(allowed)}, got: '{v}'"
            )
        return v


# Valid
t = AgentTaskAns2(
    task_id="test-001",
    task_type="research",
    instruction="Research X",
    context="Y",
    expected_output="Z",
)
print(f"Valid: task_type='{t.task_type}'")

print()

# Invalid
try:
    AgentTaskAns2(
        task_id="test-002",
        task_type="code-generation",
        instruction="Write code",
        context="Y",
        expected_output="Z",
    )
except ValidationError as e:
    print("ValidationError caught (expected):")
    for err in e.errors():
        print(f"  {err['msg']}")

### Exercise 3 — Multi-task decomposition

**Approach:**  
Create a `PlannerMultiTask` schema that returns a `list[AgentTask]`. Store as `task_dicts: list[dict]` in state. Add two sequential executor nodes, each consuming one task dict, then a synthesizer that merges both `execution_results`.

**Alternative approach:** Use LangGraph's parallel branch pattern — `add_edge("planner", "executor_a")` and `add_edge("planner", "executor_b")` simultaneously, with each executor node consuming a different index from `task_dicts`. Both branches run concurrently; the join (synthesizer) fires when both complete.

### Exercise 4 — Async concurrent execution

**Key observations from timing:**  
- Sequential: roughly `N × latency` (e.g., 2 × 3s = 6s)
- Concurrent with `asyncio.gather`: close to `max(latency_a, latency_b)` (e.g., ~3s)
- Real speedup depends on API concurrency limits and network conditions
- In Jupyter, `await` works at the top level because the notebook's event loop is already running
- In a Python script (not a notebook), wrap with `asyncio.run(run_concurrent())` instead of bare `await`